# BBDE 301366 — DQ Check

## Purpose
Checks data availability for all source tables and columns used in the BBDE 301366 metric calculations.

## Architecture
BBDE 301366 is the **aggregate segment** that unions customers from 15+ LOB source tables:
- **Retail (CPB):** EDB, PL, RESL, CC, PSI
- **Business:** SBB Credit/Deposit, COM Credit/Deposit
- **Wealth/Insurance:** TDIS/TOIS
- **TDW:** DL (Direct Lending), PB (Personal Banking)
- **TDS:** Global Markets (201037), Trade Finance (201042)

## Metrics Covered
| Notebook | Metrics | Result |
|---------|---------|--------|
| Segment | SD1 | 1,657,852 |
| Segment | CDE 1.2 (segment count) | 2,153,729 |
| Segment | SD3 (non-resident) | 235,995 |
| Segment | 6.5A (period start) | 1,545,732 |
| Segment | 6.5B (period end) | ~1,530,366 |
| Segment | 1.6 (sanctions) | ~14,238 |
| Segment | SD4 (occupation) | ~88,217 |
| Segment | SD5 (legal structure) | ~6,785 |
| Segment | 4.1A (accounts) | ~372,641 |
| Segment | 5.1 (branches) | 1,050 |
| Segment | 5.6–5.8 (sanctions jurisdictions) | 0/0/8 |
| Centralized | CDE 1.3 (Tier 1/2) | 20,928 |
| Centralized | CDE 1.2 (High risk) | *(captured)* |
| Centralized | CDE 1.4 (Medium risk) | 97,470 |
| Centralized | CDE 1.5 (Low + Unrated) | ~21,229,014 |
| Centralized | SD2 (PEP) | 20,786 |
| Centralized | CDE 1.7 (True Sanctions) | 5 |
| Centralized | CDE 1.8 (Blocked Property) | 2 |
| Centralized | SD6 (CDE SD 6 LYR) | 67,118 |
| Centralized | CDE 3.17 (UTR) | 12,434 |
| Centralized | CDE 3.18 (STR/SAR) | 5,050 |
| Centralized | CDE 3.19 (LCTR) | 107,741 |

## Pattern
Follows the TDI DQ check approach: one check per (table, column), recording which CDEs use each column.

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/GAMLConnections

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/Settings

In [ ]:
from datetime import date

# DQ results table
TABLE_NAME = SNAPSHOT_CATALOGUE_NAME + '.' + TABLE_NAME_DATA_AVA_SEG
today_date = str(date.today())

lob_id = '301366'
lob_desc = 'BBDE - Combined Segment'

def insertDQTable(SNAPSHOT_CATALOGUE_NAME, TABLE_NAME_DATA_AVA_SEG,
                  lob_id, cde_no, cde_desc, source, src_table_name,
                  data_element, availability_pct, today_date):
    query = ("INSERT INTO " + SNAPSHOT_CATALOGUE_NAME + "." + TABLE_NAME_DATA_AVA_SEG
             + " VALUES('" + lob_id + "','" + cde_no + "','" + cde_desc + "','"
             + source + "','" + src_table_name + "','" + data_element + "','"
             + str(availability_pct) + "','" + today_date + "')")
    spark.sql(query)
    return True

def check_column(cde_no, cde_desc, source, src_table, column):
    """Check availability of a column and insert into DQ table."""
    try:
        query = f"""
            SELECT count(1) as total,
                   count(CASE WHEN cast({column} as STRING) IS NOT NULL
                         AND trim(cast({column} as STRING)) != '' THEN 1 END) as valid
            FROM {src_table}
        """
        result = spark.sql(query).collect()[0]
        total, valid = result[0], result[1]
        pct = round(100.0 * valid / total, 2) if total > 0 else 0
        insertDQTable(SNAPSHOT_CATALOGUE_NAME, TABLE_NAME_DATA_AVA_SEG,
                      lob_id, cde_no, cde_desc, source, src_table,
                      column, pct, today_date)
        print(f"  {column}: {valid}/{total} = {pct}%")
    except Exception as e:
        print(f"  {column}: ERROR - {str(e)}")

print(f'DQ Check for {lob_id} ({lob_desc})')
print(f'Results table: {TABLE_NAME}')
print(f'Run date: {today_date}')

---
## Segment Data Sources — Retail LOBs
These are the retail LOB source tables that are UNIONed for the segment-level metrics.
Primary column for most metrics is `cust_id` (COUNT DISTINCT).

In [ ]:
# ============================================================
# Source: RA_FY_2025.edb_full_gbl_business
# Everyday Banking — used in SD1, SD3, 6.5A, 6.5B, 1.6, SD4, SD5
# Filters: cbs_effectv_dt = '2025-10-31', acct_type_cd = 2235, lfstgy_cd = 115
# ============================================================
print('RA_FY_2025.edb_full_gbl_business')
print('=' * 60)

check_column('SD1,SD3,6.5A,6.5B,1.6,SD4,SD5', 'Segment', 'EDW',
             'RA_FY_2025.edb_full_gbl_business', 'cast_id')

check_column('SD3,1.6', 'Segment', 'EDW',
             'RA_FY_2025.edb_full_gbl_business', 'cbs_effectv_dt')

check_column('SD3', 'Segment', 'EDW',
             'RA_FY_2025.edb_full_gbl_business', 'acct_type_cd')

check_column('SD4', 'Segment', 'EDW',
             'RA_FY_2025.edb_full_gbl_business', 'occupation_code')

check_column('SD5', 'Segment', 'EDW',
             'RA_FY_2025.edb_full_gbl_business', 'entity_type')

In [ ]:
# ============================================================
# Source: RA_FY_2025.PSI_FULL_GBL_BUSINESS
# Personal Savings & Investment
# Filters: abs_lfstgy_cd = 115, abs_effectv_dt between '2024-11-01' and '2025-10-31'
# ============================================================
print('RA_FY_2025.PSI_FULL_GBL_BUSINESS')
print('=' * 60)

check_column('SD1,SD3,6.5A,6.5B,1.6,SD4,SD5', 'Segment', 'EDW',
             'RA_FY_2025.PSI_FULL_GBL_BUSINESS', 'cast_id')

check_column('SD3,1.6', 'Segment', 'EDW',
             'RA_FY_2025.PSI_FULL_GBL_BUSINESS', 'abs_effectv_dt')

check_column('SD1,SD3', 'Segment', 'EDW',
             'RA_FY_2025.PSI_FULL_GBL_BUSINESS', 'abs_lfstgy_cd')

In [ ]:
# ============================================================
# Source: RA_FY_2025.PL_FULL_GBL
# Personal Lending
# Filters: cbs_effectv_dt = '2025-10-31'
# ============================================================
print('RA_FY_2025.PL_FULL_GBL')
print('=' * 60)

check_column('SD1,SD3,6.5A,6.5B,1.6,SD4,SD5', 'Segment', 'EDW',
             'RA_FY_2025.PL_FULL_GBL', 'cast_id')

check_column('SD3,1.6', 'Segment', 'EDW',
             'RA_FY_2025.PL_FULL_GBL', 'cbs_effectv_dt')

In [ ]:
# ============================================================
# Source: RA_FY_2025.RESL_FULL_GBL
# Real Estate Secured Lending
# Filters: cbs_effectv_dt = '2025-10-31', lfstgy_cd = 115
# ============================================================
print('RA_FY_2025.RESL_FULL_GBL')
print('=' * 60)

check_column('SD1,SD3,6.5A,6.5B,1.6,SD4,SD5', 'Segment', 'EDW',
             'RA_FY_2025.RESL_FULL_GBL', 'cas_cust_id')

check_column('SD3,1.6', 'Segment', 'EDW',
             'RA_FY_2025.RESL_FULL_GBL', 'cbs_effectv_dt')

check_column('RESL filter', 'Segment', 'EDW',
             'RA_FY_2025.RESL_FULL_GBL', 'lfstgy_cd')

In [ ]:
# ============================================================
# Source: RA_FY_2025.CC_FULL_GBL
# Credit Cards
# Filters: cbs_effectv_dt = '2025-10-31'
# ============================================================
print('RA_FY_2025.CC_FULL_GBL')
print('=' * 60)

check_column('SD1,SD3,6.5A,6.5B,1.6,SD4,SD5', 'Segment', 'EDW',
             'RA_FY_2025.CC_FULL_GBL', 'cas_cust_id')

check_column('SD3,1.6', 'Segment', 'EDW',
             'RA_FY_2025.CC_FULL_GBL', 'cbs_effectv_dt')

---
## Segment Data Sources — Business Banking LOBs

In [ ]:
# ============================================================
# Source: RA_FY_2025.sbb_credit_2025
# Small Business Banking — Credit
# ============================================================
print('RA_FY_2025.sbb_credit_2025')
print('=' * 60)

check_column('SD1,SD3,6.5A,6.5B,1.6,SD4,SD5', 'Segment', 'EDW',
             'RA_FY_2025.sbb_credit_2025', 'cif_business_customer_key')

check_column('4.1A', 'Segment', 'EDW',
             'RA_FY_2025.sbb_credit_2025', 'cdf_account_key')

In [ ]:
# ============================================================
# Source: RA_FY_2025.SBB_DP_Final
# Small Business Banking — Deposits
# ============================================================
print('RA_FY_2025.SBB_DP_Final')
print('=' * 60)

check_column('SD1,SD3,6.5A,6.5B,1.6,SD4,SD5', 'Segment', 'EDW',
             'RA_FY_2025.SBB_DP_Final', 'cif_business_customer_key')

check_column('4.1A', 'Segment', 'EDW',
             'RA_FY_2025.SBB_DP_Final', 'cdf_account_key')

In [ ]:
# ============================================================
# Source: RA_FY_2025.com_credit_2025
# Commercial Banking — Credit
# ============================================================
print('RA_FY_2025.com_credit_2025')
print('=' * 60)

check_column('SD1,SD3,6.5A,6.5B,1.6,SD4,SD5', 'Segment', 'EDW',
             'RA_FY_2025.com_credit_2025', 'cif_business_customer_key')

In [ ]:
# ============================================================
# Source: RA_FY_2025.COM_DP_Final
# Commercial Banking — Deposits
# ============================================================
print('RA_FY_2025.COM_DP_Final')
print('=' * 60)

check_column('SD1,SD3,6.5A,6.5B,1.6,SD4,SD5', 'Segment', 'EDW',
             'RA_FY_2025.COM_DP_Final', 'cif_business_customer_key')

---
## Segment Data Sources — Wealth / Insurance LOBs

In [ ]:
# ============================================================
# Source: RA_FY_2025.TDIS_FULL_GBL_BUSINESS
# TD Investment Services
# Filters: abs_effectv_dt between '2024-11-01' and '2025-10-31'
#          abs_lfstgy_cd IN (114, 115, 117)
# ============================================================
print('RA_FY_2025.TDIS_FULL_GBL_BUSINESS')
print('=' * 60)

check_column('SD1,SD3,6.5A,6.5B,1.6,SD4,SD5', 'Segment', 'EDW',
             'RA_FY_2025.TDIS_FULL_GBL_BUSINESS', 'ca_cust_id')

check_column('Filter', 'Segment', 'EDW',
             'RA_FY_2025.TDIS_FULL_GBL_BUSINESS', 'abs_effectv_dt')

check_column('Filter', 'Segment', 'EDW',
             'RA_FY_2025.TDIS_FULL_GBL_BUSINESS', 'abs_lfstgy_cd')

---
## Segment Data Sources — TDW (TD Waterhouse)

In [ ]:
# ============================================================
# Source: RA_FY_2025.views_data_all_account (TDW DL)
# TD Waterhouse — Direct Lending / DI
# Used via mstr_rec_id → cust_id conversion
# ============================================================
print('RA_FY_2025.views_data_all_account (TDW DL/DI)')
print('=' * 60)

check_column('SD1,SD3,6.5A,6.5B', 'Segment-TDW', 'EDW',
             'RA_FY_2025.views_data_all_account', 'mstr_rec_id')

check_column('SD1,SD3,6.5A,6.5B', 'Segment-TDW', 'EDW',
             'RA_FY_2025.views_data_all_account', 'bor_account_id')

---
## Segment Data Sources — TDS (TD Securities)

In [ ]:
# ============================================================
# Source: RA_FY25_VIEW.TDS_201037
# TDS Global Markets — Retail, Wealth, Commercial, FX
# Primary: ROOT_GT_ID (used as cust_id)
# ============================================================
print('RA_FY25_VIEW.TDS_201037')
print('=' * 60)

check_column('SD1,SD3,6.5A,6.5B,1.6', 'Segment-TDS', 'TDS',
             'RA_FY25_VIEW.TDS_201037', 'ROOT_GT_ID')

check_column('1.2,1.3,1.4,1.5', 'Centralized-TDS', 'TDS',
             'RA_FY25_VIEW.TDS_201037', 'RISK_LEVEL')

check_column('SD1', 'Segment-TDS', 'TDS',
             'RA_FY25_VIEW.TDS_201037', 'GT_ACCT_ID')

In [ ]:
# ============================================================
# Source: RA_FY25_VIEW.TDS_201042
# TDS Canada — Global Transaction Banking (Trade Finance)
# Primary: ROOT_GT_ID
# ============================================================
print('RA_FY25_VIEW.TDS_201042')
print('=' * 60)

check_column('SD1,SD3,6.5A,6.5B,1.6', 'Segment-TDS', 'TDS',
             'RA_FY25_VIEW.TDS_201042', 'ROOT_GT_ID')

check_column('1.2,1.3,1.4,1.5', 'Centralized-TDS', 'TDS',
             'RA_FY25_VIEW.TDS_201042', 'RISK_LEVEL')

---
## Segment — Account-Level Metric (4.1A)
**Note:** 4.1A is the only segment metric that uses `acct_id` instead of `cust_id`.

In [ ]:
# ============================================================
# 4.1A uses acct_id (COUNT DISTINCT) across all LOBs
# Check account key availability per LOB source
# ============================================================
print('4.1A — Account-level primary key checks')
print('=' * 60)

# EDB account
check_column('4.1A', 'Segment', 'EDW',
             'RA_FY_2025.edb_full_gbl_business', 'acct_id')

# PL account
check_column('4.1A', 'Segment', 'EDW',
             'RA_FY_2025.PL_FULL_GBL', 'acct_id')

# RESL account
check_column('4.1A', 'Segment', 'EDW',
             'RA_FY_2025.RESL_FULL_GBL', 'acct_id')

# CC account
check_column('4.1A', 'Segment', 'EDW',
             'RA_FY_2025.CC_FULL_GBL', 'acct_id')

---
## Branch Data Sources (5.1, 5.6–5.9)
Branch metrics use `count(1)` or `count(lien)` — not customer-based.

In [ ]:
# ============================================================
# Source: ra_adido_2025.bbde_branches_fy2025
# BBDE branches for 5.1 (count of branches)
# ============================================================
print('ra_adido_2025.bbde_branches_fy2025')
print('=' * 60)

# 5.1 uses count(1) — check row existence and key columns
check_column('5.1', 'Branch', 'ADIDO',
             'ra_adido_2025.bbde_branches_fy2025', 'AU')

check_column('5.1', 'Branch', 'ADIDO',
             'ra_adido_2025.bbde_branches_fy2025', 'Branch_Number')

In [ ]:
# ============================================================
# Source: ra_adido_2025.tds_branches_location_FY25
# TDS branches for 5.6, 5.7, 5.8, 5.9 (count(lien))
# Only applicable for TDS (201037 + 201042)
# ============================================================
print('ra_adido_2025.tds_branches_location_FY25')
print('=' * 60)

check_column('5.6,5.7,5.8,5.9', 'Branch-TDS', 'ADIDO',
             'ra_adido_2025.tds_branches_location_FY25', 'lien')

check_column('5.6,5.7,5.8', 'Branch-TDS', 'ADIDO',
             'ra_adido_2025.tds_branches_location_FY25', 'CountryCode')

check_column('5.6,5.7,5.8', 'Branch-TDS', 'ADIDO',
             'ra_adido_2025.tds_branches_location_FY25', 'TDS_201037')

check_column('5.6,5.7,5.8', 'Branch-TDS', 'ADIDO',
             'ra_adido_2025.tds_branches_location_FY25', 'TDS_201042')

---
## Reference / Lookup Tables

In [ ]:
# ============================================================
# Source: ra_adido_2025.sanctions_country_risk_rating_2025
# Sanctions risk ratings for branch metrics 5.6, 5.7, 5.8
# ============================================================
print('ra_adido_2025.sanctions_country_risk_rating_2025')
print('=' * 60)

check_column('5.6,5.7,5.8', 'Reference', 'ADIDO',
             'ra_adido_2025.sanctions_country_risk_rating_2025', 'ISO_ALPHA2')

check_column('5.6,5.7,5.8', 'Reference', 'ADIDO',
             'ra_adido_2025.sanctions_country_risk_rating_2025', 'RiskRating')

---
## Centralized — CRR (Customer Risk Rating)
Used for CDE 1.2 (High), CDE 1.3 (Tier 1/2), CDE 1.4 (Medium), CDE 1.5 (Low + Unrated)

In [ ]:
# ============================================================
# Source: rafy2025_centralized1.scorable_rated_ca
# CRR rated customers — primary source for centralized metrics
# Key derivation: substr(v_entity_id, -9, 9) AS cust_no
# ============================================================
print('rafy2025_centralized1.scorable_rated_ca')
print('=' * 60)

check_column('1.2,1.3,1.4,1.5', 'Centralized', 'CRR',
             'rafy2025_centralized1.scorable_rated_ca', 'v_entity_id')

check_column('1.2,1.3,1.4,1.5', 'Centralized', 'CRR',
             'rafy2025_centralized1.scorable_rated_ca', 'risk_rating')

check_column('1.5', 'Centralized', 'CRR',
             'rafy2025_centralized1.scorable_rated_ca', 'v_cust_type_cd')

In [ ]:
# ============================================================
# Centralized metrics also join the same retail LOB tables
# with the CRR data. Check the join columns per LOB:
# Pattern: ON h.cust_no = r.cust_no
# where cust_no = substr(v_entity_id, -9, 9)
# ============================================================
print('Centralized LOB join columns')
print('=' * 60)

# EDB — cust_no is derived from cast_id
check_column('1.2,1.3,1.4,1.5', 'Centralized-EDB', 'EDW',
             'RA_FY_2025.edb_full_gbl_business', 'cust_no')

# PSI — cust_no from PSI data
check_column('1.2,1.3,1.4,1.5', 'Centralized-PSI', 'EDW',
             'RA_FY_2025.PSI_FULL_GBL_BUSINESS', 'cust_no')

# PL — cust_no
check_column('1.2,1.3,1.4,1.5', 'Centralized-PL', 'EDW',
             'RA_FY_2025.PL_FULL_GBL', 'cust_no')

# RESL — cas_cust_id as primary
check_column('1.2,1.3,1.4,1.5', 'Centralized-RESL', 'EDW',
             'RA_FY_2025.RESL_FULL_GBL', 'cust_no')

# CC — cust_no
check_column('1.2,1.3,1.4,1.5', 'Centralized-CC', 'EDW',
             'RA_FY_2025.CC_FULL_GBL', 'cust_no')

In [ ]:
# ============================================================
# Business customer CRR join columns
# Pattern: right(cent.v_entity_id, 8) = right(main.cif_business_customer_key, 8)
# Filter: cent.v_entity_id LIKE 'CIF-00415'
# ============================================================
print('Centralized — Business customer key checks')
print('=' * 60)

check_column('1.2,1.3,1.4,1.5', 'Centralized-SBB', 'EDW',
             'RA_FY_2025.SBB_DP_Final', 'cif_business_customer_key')

check_column('1.2,1.3,1.4,1.5', 'Centralized-COM', 'EDW',
             'RA_FY_2025.COM_DP_Final', 'cif_business_customer_key')

check_column('1.2,1.3,1.4,1.5', 'Centralized-SBB-Credit', 'EDW',
             'RA_FY_2025.sbb_credit_2025', 'cif_business_customer_key')

check_column('1.2,1.3,1.4,1.5', 'Centralized-COM-Credit', 'EDW',
             'RA_FY_2025.com_credit_2025', 'cif_business_customer_key')

---
## Centralized — TDW/TDS Dedup Infrastructure
The centralized metrics use a complex dedup chain to avoid double-counting customers
that appear in both retail and TDW/TDS.

In [ ]:
# ============================================================
# Source: ra_adido_2025.bbde_retail1_common_client_final
# Dedup table — maps retail customers to TDW/TDS
# ~157,666 common customers
# ============================================================
print('ra_adido_2025.bbde_retail1_common_client_final')
print('=' * 60)

check_column('1.2,1.3,1.4,1.5,SD2', 'Dedup', 'ADIDO',
             'ra_adido_2025.bbde_retail1_common_client_final', 'cust_id')

check_column('1.2,1.3,1.4,1.5,SD2', 'Dedup', 'ADIDO',
             'ra_adido_2025.bbde_retail1_common_client_final', 'edw_customer_id')

In [ ]:
# ============================================================
# Source: ra_adido_2025.tds_retail_common_clients
# TDS retail common dedup (REPLACE(Customer_Identifier_ID, '.', ''))
# ============================================================
print('ra_adido_2025.tds_retail_common_clients')
print('=' * 60)

check_column('1.2,1.3,1.4,1.5,SD2', 'Dedup-TDS', 'ADIDO',
             'ra_adido_2025.tds_retail_common_clients', 'Customer_Identifier_ID')

---
## Centralized — SD2 (PEP — Politically Exposed Persons)
SD2 uses the PEP data source instead of CRR.
Name matching via `upper(concat(first_nm, ' ', last_nm))` as a secondary identifier.

In [ ]:
# ============================================================
# Source: ra_adido_2025.pep_list_2025_exploded
# PEP list for SD2
# Key: substring(entity, -9, 9) AS cust_no
# ============================================================
print('ra_adido_2025.pep_list_2025_exploded')
print('=' * 60)

check_column('SD2', 'PEP', 'ADIDO',
             'ra_adido_2025.pep_list_2025_exploded', 'entity')

# Name columns used for fullname matching
check_column('SD2', 'PEP', 'ADIDO',
             'ra_adido_2025.pep_list_2025_exploded', 'first_nm')

check_column('SD2', 'PEP', 'ADIDO',
             'ra_adido_2025.pep_list_2025_exploded', 'last_nm')

In [ ]:
# ============================================================
# Source: ra_adido_2025.peb_pep_estrace_2025
# PEP extract for TDS — joins on Account_GoldTier_ID
# ============================================================
print('ra_adido_2025.peb_pep_estrace_2025')
print('=' * 60)

check_column('SD2', 'PEP-TDS', 'ADIDO',
             'ra_adido_2025.peb_pep_estrace_2025', 'Account_GoldTier_ID')

---
## Centralized — Classification / Legal Structures
Used in TDS risk rating derivation for centralized metrics.

In [ ]:
# ============================================================
# Source: RA_FY25.CLASSIFICATION / LEGAL_STRUCTURES
# TDS entity classification — Updated_Risk_Rating derivation
# ============================================================
print('RA_FY25 Classification / Legal Structures')
print('=' * 60)

check_column('1.2,1.3,1.4,1.5', 'Centralized-TDS', 'TDS',
             'ra_adido_2025.entity_ref_list_ca2025', 'Entity_Type_Code')

check_column('1.2,1.3,1.4,1.5', 'Centralized-TDS', 'TDS',
             'ra_adido_2025.entity_ref_list_ca2025', 'Updated_Risk_Rating')

---
## EDW Customer Relationship Table
Used for TDW customer dedup (mstr_rec_id ↔ bor_account_id conversion).

In [ ]:
# ============================================================
# Source: ra_fy_2025.edw_customer (relationship table)
# Maps retail cust_id → EDW bor_account_id → mstr_rec_id
# ~21,869,742 rows (distinct: 11,988,358)
# ============================================================
print('EDW Customer Relationship')
print('=' * 60)

check_column('1.2,1.3,1.4,1.5,SD2', 'Dedup-EDW', 'EDW',
             'ra_fy_2025.edw_customer', 'edw_customer_id')

check_column('1.2,1.3,1.4,1.5,SD2', 'Dedup-EDW', 'EDW',
             'ra_fy_2025.edw_customer', 'bor_account_id')

---
## Centralized — CDE 1.7 (True Sanctions Match)
Source: `ra_adido_2025.true_sanction_match_2025`
Key: `substring(Customer, -9, 9)` for retail; `CustomerType = 'Non-Personal'` for business.
Name matching: `upper(concat(first_nm, ' ', last_nm))` = fulln.
TDS uses **hardcoded** `ROOT_LEGAL_NAME LIKE` filters (specific sanctioned entity names).
**Result: 5 customers**

In [ ]:
# ============================================================
# Source: ra_adido_2025.true_sanction_match_2025
# CDE 1.7 — True sanctions match
# Key: substring(Customer, -9, 9) AS c_no
# Business: CustomerType = 'Non-Personal'
# ============================================================
print('CDE 1.7 — ra_adido_2025.true_sanction_match_2025')
print('=' * 60)

check_column('1.7', 'Sanctions', 'ADIDO',
             'ra_adido_2025.true_sanction_match_2025', 'Customer')

check_column('1.7', 'Sanctions', 'ADIDO',
             'ra_adido_2025.true_sanction_match_2025', 'CustomerType')

---
## Centralized — CDE 1.8 (Sanctioned / Blocked Property)
Source: `ra_adido_2025.customer_sanctioned_blocked_property_2025`
Retail join: `CUSTOMERIMPACTED = upper(concat(first_nm, ' ', last_nm))`
Business join: `left(ACCOUNTBLOCKED, 7) = CASE WHEN account_key=7 THEN ... ELSE right(...,7)`
TDS uses hardcoded `ROOT_LEGAL_NAME LIKE` filters.
**Result: 2 customers**

In [ ]:
# ============================================================
# Source: ra_adido_2025.customer_sanctioned_blocked_property_2025
# CDE 1.8 — Sanctioned / blocked property
# Retail: CUSTOMERIMPACTED (name-based)
# Business: ACCOUNTBLOCKED (account-based)
# ============================================================
print('CDE 1.8 — ra_adido_2025.customer_sanctioned_blocked_property_2025')
print('=' * 60)

check_column('1.8', 'Sanctions', 'ADIDO',
             'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'CUSTOMERIMPACTED')

check_column('1.8', 'Sanctions', 'ADIDO',
             'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'ACCOUNTBLOCKED')

---
## Centralized — SD6 (CDE SD 6 LYR)
Source: `rafy2025_centralized1.cde_sd_6_lyr_fy2025`
Key: `substring(v_entity_id, -9, 9)` for CIF; `REPLACE(v_entity_id, 'HSC-', '')` for TDW.
TDS filter: `date(ROOT_OPEN_DATE) > date('2024-10-31')` (date-based, not risk-level).
**Result: 67,118 customers**

In [ ]:
# ============================================================
# Source: rafy2025_centralized1.cde_sd_6_lyr_fy2025
# SD6 — CDE SD 6 LYR (same CIF key derivation as CRR)
# TDS uses date(ROOT_OPEN_DATE) > date('2024-10-31')
# ============================================================
print('SD6 — rafy2025_centralized1.cde_sd_6_lyr_fy2025')
print('=' * 60)

check_column('SD6', 'Centralized', 'Centralized',
             'rafy2025_centralized1.cde_sd_6_lyr_fy2025', 'v_entity_id')

---
## Centralized — CDE 3.17 (UTR — Unusual Transaction Report)
Source: `rafy2025_centralized1.TD_UTR_CDE_3_17_2025_Cust_details`
Three-way join: `cust_no + birth_dt + full_name`.
Business: `1pad(right(main.cif_business_customer_key, 8), 9, '0')`, filter `account_type <> 'TDM'`.
TDW DI & PB: NOT applicable.
TDS: direct join `ROOT_GT_ID = utr.cust_no`.
**Result: 12,434 customers**

In [ ]:
# ============================================================
# Source: rafy2025_centralized1.TD_UTR_CDE_3_17_2025_Cust_details
# CDE 3.17 — UTR (Unusual Transaction Report)
# Retail: cust_no + cust_type_mn + birth_dt + full_name
# Business: lpad(right(cif_business_customer_key, 8), 9, '0')
# TDW: NOT applicable  |  TDS: ROOT_GT_ID = utr.cust_no
# ============================================================
print('CDE 3.17 — rafy2025_centralized1.TD_UTR_CDE_3_17_2025_Cust_details')
print('=' * 60)

check_column('3.17', 'UTR', 'Centralized',
             'rafy2025_centralized1.TD_UTR_CDE_3_17_2025_Cust_details', 'cust_no')

check_column('3.17', 'UTR', 'Centralized',
             'rafy2025_centralized1.TD_UTR_CDE_3_17_2025_Cust_details', 'cust_type_mn')

check_column('3.17', 'UTR', 'Centralized',
             'rafy2025_centralized1.TD_UTR_CDE_3_17_2025_Cust_details', 'birth_dt')

check_column('3.17', 'UTR', 'Centralized',
             'rafy2025_centralized1.TD_UTR_CDE_3_17_2025_Cust_details', 'full_name')

check_column('3.17', 'UTR', 'Centralized',
             'rafy2025_centralized1.TD_UTR_CDE_3_17_2025_Cust_details', 'account_type')

---
## Centralized — CDE 3.18 (STR/SAR — Suspicious Transaction Report)
Source: `rafy2025_centralized1.td_sar_cde_3_18_2025`
Retail join: `STR_Admin_Matched_CustomerID` + `STR_Admin_Matched_Customer_Type`.
Business: `lpad(right(cif_business_customer_key, 8), 9, '0')`, filter `account_type <> 'TDM'`.
TDS: `GT_ACCT_ID = substr(STR_LETUA_Customer_Account_Number, 0, instr(..., '.') - 1)`.
TDW PB: NOT applicable.
**Result: 5,050 customers**

In [ ]:
# ============================================================
# Source: rafy2025_centralized1.td_sar_cde_3_18_2025
# CDE 3.18 — STR/SAR (Suspicious Transaction Report)
# Retail: STR_Admin_Matched_CustomerID + STR_Admin_Matched_Customer_Type
# TDS: STR_LETUA_Customer_Account_Number
# ============================================================
print('CDE 3.18 — rafy2025_centralized1.td_sar_cde_3_18_2025')
print('=' * 60)

check_column('3.18', 'STR/SAR', 'Centralized',
             'rafy2025_centralized1.td_sar_cde_3_18_2025', 'STR_Admin_Matched_CustomerID')

check_column('3.18', 'STR/SAR', 'Centralized',
             'rafy2025_centralized1.td_sar_cde_3_18_2025', 'STR_Admin_Matched_Customer_Type')

check_column('3.18', 'STR/SAR', 'Centralized',
             'rafy2025_centralized1.td_sar_cde_3_18_2025', 'STR_LETUA_Customer_Account_Number')

check_column('3.18', 'STR/SAR', 'Centralized',
             'rafy2025_centralized1.td_sar_cde_3_18_2025', 'Customer_Address_PostalCode')

check_column('3.18', 'STR/SAR', 'Centralized',
             'rafy2025_centralized1.td_sar_cde_3_18_2025', 'customer_first_name')

check_column('3.18', 'STR/SAR', 'Centralized',
             'rafy2025_centralized1.td_sar_cde_3_18_2025', 'customer_last_name')

---
## Centralized — CDE 3.19 (LCTR — Large Cash Transaction Report)
Source: `rafy2025_centralized1.cde_3_19_lctr`
Retail join: `account_number = acct_no`.
Business: `lpad(cent.account, 7, '0')` + `right(d1.account_key, 7)`.
TDS: partial name match `substring(replace(ROOT_LEGAL_NAME,' ',''),1,3) = substring(replace(client_name,' ',''),1,3)`.
TDW DI & PB: NOT applicable. No TDS TF queries.
**Result: 107,741 customers**

In [ ]:
# ============================================================
# Source: rafy2025_centralized1.cde_3_19_lctr
# CDE 3.19 — LCTR (Large Cash Transaction Report)
# Retail: account_number = acct_no
# Business: lpad(cent.account, 7, '0') + right(d1.account_key, 7)
# TDS: partial name match on ROOT_LEGAL_NAME
# ============================================================
print('CDE 3.19 — rafy2025_centralized1.cde_3_19_lctr')
print('=' * 60)

check_column('3.19', 'LCTR', 'Centralized',
             'rafy2025_centralized1.cde_3_19_lctr', 'account_number')

check_column('3.19', 'LCTR', 'Centralized',
             'rafy2025_centralized1.cde_3_19_lctr', 'client_name')

---
## Summary — Row Counts per Source Table
Quick check of total rows available in each key source table.

In [ ]:
# ============================================================
# Row count summary for all source tables
# ============================================================
tables = [
    ('RA_FY_2025.edb_full_gbl_business', 'EDB'),
    ('RA_FY_2025.PSI_FULL_GBL_BUSINESS', 'PSI'),
    ('RA_FY_2025.PL_FULL_GBL', 'PL'),
    ('RA_FY_2025.RESL_FULL_GBL', 'RESL'),
    ('RA_FY_2025.CC_FULL_GBL', 'CC'),
    ('RA_FY_2025.sbb_credit_2025', 'SBB Credit'),
    ('RA_FY_2025.SBB_DP_Final', 'SBB Deposit'),
    ('RA_FY_2025.com_credit_2025', 'COM Credit'),
    ('RA_FY_2025.COM_DP_Final', 'COM Deposit'),
    ('RA_FY_2025.TDIS_FULL_GBL_BUSINESS', 'TDIS'),
    ('RA_FY25_VIEW.TDS_201037', 'TDS 201037'),
    ('RA_FY25_VIEW.TDS_201042', 'TDS 201042'),
    ('ra_adido_2025.bbde_branches_fy2025', 'BBDE Branches'),
    ('ra_adido_2025.tds_branches_location_FY25', 'TDS Branches'),
    ('rafy2025_centralized1.scorable_rated_ca', 'CRR Rated'),
    ('ra_adido_2025.pep_list_2025_exploded', 'PEP List'),
    ('ra_adido_2025.bbde_retail1_common_client_final', 'Retail-TDW Common'),
    ('ra_adido_2025.tds_retail_common_clients', 'TDS Retail Common'),
    ('ra_adido_2025.true_sanction_match_2025', 'True Sanctions'),
    ('ra_adido_2025.customer_sanctioned_blocked_property_2025', 'Blocked Property'),
    ('rafy2025_centralized1.cde_sd_6_lyr_fy2025', 'SD6 LYR'),
    ('rafy2025_centralized1.TD_UTR_CDE_3_17_2025_Cust_details', 'UTR 3.17'),
    ('rafy2025_centralized1.td_sar_cde_3_18_2025', 'STR/SAR 3.18'),
    ('rafy2025_centralized1.cde_3_19_lctr', 'LCTR 3.19'),
    ('RA_FY_2025.edw_customer', 'EDW Customer Relation'),
    ('RA_FY_2025.views_data_all_account', 'TDW All Accounts'),
    ('ra_adido_2025.peb_pep_estrace_2025', 'PEP TDS Estrace'),
    ('ra_adido_2025.entity_ref_list_ca2025', 'TDS Classification'),
    ('ra_adido_2025.sanctions_country_risk_rating_2025', 'Sanctions Reference'),
]

print(f'{"Table":<55} {"LOB":<20} {"Row Count":>12}')
print('=' * 90)
for table, lob in tables:
    try:
        cnt = spark.sql(f'SELECT count(1) FROM {table}').collect()[0][0]
        print(f'{table:<55} {lob:<20} {cnt:>12,}')
    except Exception as e:
        print(f'{table:<55} {lob:<20} {"ERROR":>12}')

---
## Metrics Not Applicable
The following metrics are **N/A** for BBDE 301366 and do not require DQ checks:

| Metric | Reason |
|--------|--------|
| 6.1, 6.2, 6.3, 6.4, 6.6, 6.6B | Not calculated in any of the AU's of CPB, TDI, and TDW |
| 5.9 | Special: in absence of high crime rate area list, all branches counted |

## TDW / TDS Applicability Notes
| Metric | TDW DI/PB | TDS TF |
|--------|-----------|--------|
| CDE 3.17 (UTR) | NOT applicable | N/A |
| CDE 3.18 (STR/SAR) | PB NOT applicable | N/A |
| CDE 3.19 (LCTR) | NOT applicable | No TDS TF queries |